# SurGeet AI — Pro GPU Training (Free, RVC-style Voice Clone)

Real neural voice training on a **free GPU** (Kaggle: 30 hrs/week free GPU, or Google Colab free tier).
Pipeline: GPU check → install Applio (RVC) → upload dataset → preprocess → RMVPE pitch + HuBERT features → train → download `.pth` + `.index`.

**Dataset:** 3–10 minutes of clean, dry, single-voice singing/speech as `dataset.zip` (wav/mp3 files inside). No music, no reverb.

In [ ]:
# 1) GPU check
import subprocess, sys
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'No GPU found - enable GPU: Kaggle Settings > Accelerator > GPU T4, or Colab Runtime > Change runtime type > GPU')

In [ ]:
# 2) Install Applio (RVC training framework) - takes a few minutes
import subprocess, sys, os
if not os.path.exists('Applio'):
    subprocess.run(['git','clone','https://github.com/IAHispano/Applio.git'],check=False)
%cd Applio
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=False)
print('Install step done. If requirements fail on this platform, use the Kaggle/Colab images pinned in Applio README.')

In [ ]:
# 3) Upload dataset.zip
# Kaggle: use the notebook's Upload button / add data. Colab: run this cell and pick dataset.zip
try:
    from google.colab import files
    up = files.upload()
    print('Uploaded:', list(up.keys()))
except Exception as e:
    print('Not Colab (or upload skipped):', e)
    print('On Kaggle, upload dataset.zip via the notebook sidebar instead.')

In [ ]:
# 4) Unzip dataset into the training folder
import zipfile, os, glob
zips = glob.glob('*.zip') + glob.glob('/kaggle/input/**/*.zip', recursive=True)
assert zips, 'dataset.zip not found - upload it first (cell 3)'
os.makedirs('dataset_raw', exist_ok=True)
with zipfile.ZipFile(zips[0]) as z: z.extractall('dataset_raw')
audios = glob.glob('dataset_raw/**/*.*', recursive=True)
print(len(audios), 'audio files ready')

In [ ]:
# 5) Preprocess + extract pitch (RMVPE) and features
# Applio CLI: see `python core.py --help` in your installed version - commands below follow Applio's documented flow.
import subprocess, sys
NAME = 'surgeet-voice'  # your model name
SR = '40k'               # sample rate: 32k / 40k / 48k - keep consistent
subprocess.run([sys.executable,'core.py','preprocess',NAME,'dataset_raw',SR],check=False)
subprocess.run([sys.executable,'core.py','extract',NAME,SR,'rmvpe'],check=False)
print('Preprocess + extract done for model:', NAME)

In [ ]:
# 6) Train (free-GPU friendly defaults: 200 epochs, batch 8; drop to 100 epochs if the free session times out)
import subprocess, sys
NAME='surgeet-voice'; SR='40k'
subprocess.run([sys.executable,'core.py','train',NAME,SR,'200','8'],check=False)
print('Training finished (check logs above for the saved checkpoint).')

In [ ]:
# 7) Find and download your model (.pth + .index)
import glob
outs = glob.glob('**/*.pth', recursive=True) + glob.glob('**/*.index', recursive=True)
print('Model files:'); [print(' ', o) for o in outs]
try:
    from google.colab import files
    for o in outs: files.download(o)
except Exception:
    print('On Kaggle: download the .pth and .index from the notebook Output panel.')
print('Use the .pth + .index in Applio/RVC for full voice conversion.')

## Notes
- Free tiers have session/time limits — save checkpoints often; resume by re-running from the train cell with your last checkpoint.
- Only train voices you own or have permission to clone.
- Back on surgeet-ai.github.io, the in-browser Train Model gives you an instant measured voice profile (pitch/waveform/vibrato/tempo) with no GPU wait.